# Sentiment and Emotion Analysis in English Literature

A deep learning project that detects **sentiment polarity** and **emotion categories** from literary text using a hybrid neural network architecture.

---

## Architecture

The model stacks four components in sequence:

- **BERT-base** — generates rich contextual word embeddings from the input text
- **CNN** — extracts local patterns using multi-scale filters (sizes 3, 4, 5)
- **BiLSTM** — captures long-range dependencies in both directions
- **Self-Attention** — focuses the model on emotionally significant tokens

The CNN and BiLSTM outputs are fused and fed into two separate classification heads — one for sentiment, one for emotions.

---

## What the Model Predicts

**Sentiment** (3 classes): Positive · Negative · Neutral

**Emotion** (7 classes): Joy · Sadness · Anger · Fear · Surprise · Disgust · Neutral

---

## Datasets Used

- **Abdallah Wagih Emotion Dataset** (Kaggle)
- **ISEAR Dataset** (Kaggle)
- **GoEmotions** (Google / HuggingFace)

---


## 1. Check GPU & Setup

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\nGPU ready!")
else:
    print("\nNO GPU! Go to Runtime -> Change runtime type -> GPU")
    raise RuntimeError("GPU required")

## 2. Install Dependencies

In [ ]:
%%capture
!pip install -q transformers torch torchvision
!pip install -q nltk scikit-learn matplotlib seaborn tqdm pyyaml
!pip install -q kagglehub datasets

import nltk
for pkg in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger']:
    nltk.download(pkg, quiet=True)

print("All dependencies installed!")

## 3. Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel

# 7 emotion classes (added Disgust, Neutral is now properly trained)
EMOTION_LABELS = ['Joy', 'Sadness', 'Anger', 'Fear', 'Surprise', 'Disgust', 'Neutral']
NUM_EMOTIONS   = len(EMOTION_LABELS)  # 7
NUM_SENTIMENTS = 3  # Negative / Neutral / Positive

class SelfAttention(nn.Module):
    def __init__(self, hidden_size, dropout=0.1):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, lstm_output, mask=None):
        energy = torch.tanh(self.W(lstm_output))
        scores = self.v(energy).squeeze(-1)
        if mask is not None:
            scores = scores.float()
            scores = scores.masked_fill(mask.float() == 0, -1e4)
        attention_weights = F.softmax(scores, dim=1)
        attention_weights = self.dropout(attention_weights)
        context = torch.sum(attention_weights.unsqueeze(-1) * lstm_output.float(), dim=1)
        return context, attention_weights


class EnhancedSentimentModel(nn.Module):
    def __init__(self, num_emotions=NUM_EMOTIONS, num_sentiments=NUM_SENTIMENTS):
        super().__init__()

        print("Loading BERT-base-uncased...")
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        bert_size = 768

        # Freeze first 6 BERT layers (better regularisation for small datasets)
        for i in range(6):
            for param in self.bert.encoder.layer[i].parameters():
                param.requires_grad = False

        # CNN — balanced filter sizes
        self.convs = nn.ModuleList([
            nn.Conv1d(bert_size, 512, 3),
            nn.Conv1d(bert_size, 512, 4),
            nn.Conv1d(bert_size, 512, 5)
        ])
        cnn_out = 512 * 3  # 1536

        # BiLSTM
        self.lstm = nn.LSTM(
            bert_size, 512,
            num_layers=3,
            bidirectional=True,
            dropout=0.2,
            batch_first=True
        )
        lstm_out = 512 * 2  # 1024

        self.attention = SelfAttention(lstm_out, 0.1)

        # LayerNorm each branch before fusion (stabilises scale differences)
        self.cnn_norm  = nn.LayerNorm(cnn_out)
        self.lstm_norm = nn.LayerNorm(lstm_out)

        combined = cnn_out + lstm_out  # 2560

        # Sentiment classifier
        self.sentiment_clf = nn.Sequential(
            nn.Linear(combined, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_sentiments)
        )

        # Emotion classifier
        self.emotion_clf = nn.Sequential(
            nn.Linear(combined, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_emotions)
        )

        self.dropout = nn.Dropout(0.2)

    def forward(self, input_ids, attention_mask):
        bert_out = self.bert(input_ids, attention_mask).last_hidden_state

        # CNN path
        x = bert_out.float().transpose(1, 2)
        cnn_feats = []
        for conv in self.convs:
            h = torch.relu(conv(x))
            h = torch.max_pool1d(h, h.size(2)).squeeze(2)
            cnn_feats.append(h)
        cnn_out = self.cnn_norm(torch.cat(cnn_feats, 1))
        cnn_out = self.dropout(cnn_out)

        # BiLSTM + Attention path
        lstm_out, _ = self.lstm(bert_out.float())
        lstm_out = self.dropout(lstm_out)
        context, attn_weights = self.attention(lstm_out, attention_mask)
        context = self.lstm_norm(context)

        combined = torch.cat([cnn_out, context], 1)
        sentiment = self.sentiment_clf(combined)
        emotion   = self.emotion_clf(combined)
        return sentiment, emotion, attn_weights

print("Model classes defined!")
print(f"Emotion classes: {EMOTION_LABELS}")

## 4. Download & Prepare Datasets

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
from datasets import load_dataset
from sklearn.model_selection import train_test_split

print("Downloading datasets...")
print("="*60)

# ──────────────────────────────────────────────────────────────
# Helper: map raw emotion label → (emotion_vec, sentiment_int)
# EMOTION_LABELS = ['Joy','Sadness','Anger','Fear','Surprise','Disgust','Neutral']
# ──────────────────────────────────────────────────────────────
def make_vec(idx, sent):
    vec = np.zeros(NUM_EMOTIONS, dtype=np.float32)
    vec[idx] = 1.0
    return vec, int(sent)

# ── 1. WAGIH DATASET ───────────────────────────────────────────
print("\n[1/3] Wagih Emotion Dataset...")
path_wagih = kagglehub.dataset_download("abdallahwagih/emotion-dataset")
wagih_csv  = [f for f in os.listdir(path_wagih) if f.endswith('.csv')][0]
df_wagih   = pd.read_csv(os.path.join(path_wagih, wagih_csv))
w_emo = 'Emotion' if 'Emotion' in df_wagih.columns else df_wagih.columns[1]
w_txt = 'Comment' if 'Comment' in df_wagih.columns else df_wagih.columns[0]
print(f"    Samples: {len(df_wagih)} | Labels: {df_wagih[w_emo].unique().tolist()}")

def map_wagih(row):
    lbl = str(row[w_emo]).strip().lower()
    txt = str(row[w_txt])
    if   lbl in ['joy', 'love', 'happy']:    vec, s = make_vec(0, 2)  # Joy / Positive
    elif lbl in ['sadness', 'sad']:           vec, s = make_vec(1, 0)  # Sadness / Negative
    elif lbl in ['anger', 'angry']:           vec, s = make_vec(2, 0)  # Anger / Negative
    elif lbl in ['fear', 'afraid']:           vec, s = make_vec(3, 0)  # Fear / Negative
    elif lbl in ['surprise']:                 vec, s = make_vec(4, 2)  # Surprise / Positive
    elif lbl in ['disgust']:                  vec, s = make_vec(5, 0)  # Disgust / Negative
    else:                                     vec, s = make_vec(6, 1)  # Neutral
    return pd.Series([txt, vec, s])

df_w = df_wagih.apply(map_wagih, axis=1)
df_w.columns = ['text', 'emotions', 'sentiment']
print(f"    Processed: {len(df_w)} samples")

# ── 2. ISEAR DATASET ───────────────────────────────────────────
print("\n[2/3] ISEAR Dataset...")
path_isear = kagglehub.dataset_download("faisalsanto007/isear-dataset")
isear_csv  = None
for f in os.listdir(path_isear):
    fp = os.path.join(path_isear, f)
    if f.endswith('.csv'):
        isear_csv = fp
    elif os.path.isdir(fp):
        for sf in os.listdir(fp):
            if sf.endswith('.csv'):
                isear_csv = os.path.join(fp, sf)

df_i = pd.DataFrame(columns=['text', 'emotions', 'sentiment'])
if isear_csv:
    df_isear = pd.read_csv(isear_csv, on_bad_lines='skip')
    emo_col  = 'sentiment'
    txt_col  = 'content'
    print(f"    Samples: {len(df_isear)} | Labels: {df_isear[emo_col].unique().tolist()}")

    def map_isear(row):
        lbl = str(row[emo_col]).strip().lower()
        txt = str(row[txt_col]) if pd.notna(row[txt_col]) else ""
        if len(txt) < 10:
            return None
        if   'joy'   in lbl or 'happy' in lbl: vec, s = make_vec(0, 2)
        elif 'sad'   in lbl:                    vec, s = make_vec(1, 0)
        elif 'ang'   in lbl:                    vec, s = make_vec(2, 0)
        elif 'fear'  in lbl:                    vec, s = make_vec(3, 0)
        elif 'disgust' in lbl:                  vec, s = make_vec(5, 0)  # Disgust — own class!
        elif 'shame' in lbl or 'guilt' in lbl:  vec, s = make_vec(5, 0)  # Shame → Disgust
        # ↓ FIX: neutral / unmatched rows are KEPT as Neutral, not discarded
        else:                                    vec, s = make_vec(6, 1)
        return pd.Series([txt, vec, s])

    result = df_isear.apply(map_isear, axis=1).dropna()
    result.columns = ['text', 'emotions', 'sentiment']
    df_i = result
    print(f"    Processed: {len(df_i)} samples (Neutral rows now kept!)")
else:
    print("    ERROR: No CSV found")

# ── 3. GoEmotions (Google) — best source for Neutral & Disgust ──
print("\n[3/3] GoEmotions (HuggingFace)...")
# 27-class dataset from Google; we map down to our 7 classes
GO_EMOTION_MAP = {
    # Joy
    'admiration':0,'amusement':0,'approval':0,'caring':0,'desire':0,
    'excitement':0,'gratitude':0,'joy':0,'love':0,'optimism':0,
    'pride':0,'relief':0,
    # Sadness
    'disappointment':1,'grief':1,'remorse':1,'sadness':1,
    # Anger
    'anger':2,'annoyance':2,'disapproval':2,
    # Fear
    'fear':3,'nervousness':3,
    # Surprise
    'confusion':4,'curiosity':4,'realization':4,'surprise':4,
    # Disgust
    'disgust':5,'embarrassment':5,
    # Neutral
    'neutral':6,
}
# sentiment map per emotion index
EMO_TO_SENT = {0:2, 1:0, 2:0, 3:0, 4:2, 5:0, 6:1}

try:
    go_ds = load_dataset('google-research-datasets/go_emotions', 'simplified', split='train')
    label_names = go_ds.features['labels'].feature.names  # outside loop
    rows = []
    for item in go_ds:
        txt    = item['text']
        labels = item['labels']          # list of GoEmotions label indices
        vec = np.zeros(NUM_EMOTIONS, dtype=np.float32)
        sentiments = []
        mapped_any = False
        for li in labels:
            name = label_names[li]
            if name in GO_EMOTION_MAP:
                emo_idx = GO_EMOTION_MAP[name]
                vec[emo_idx] = 1.0
                sentiments.append(EMO_TO_SENT[emo_idx])
                mapped_any = True
        if not mapped_any or len(txt) < 10:
            continue
        # Resolve sentiment: majority vote, tie → neutral
        from collections import Counter
        cnt = Counter(sentiments)
        sent = cnt.most_common(1)[0][0]
        rows.append({'text': txt, 'emotions': vec, 'sentiment': sent})
    df_go = pd.DataFrame(rows)
    print(f"    GoEmotions processed: {len(df_go):,} samples")
    neutral_count = sum(1 for r in rows if np.argmax(r['emotions']) == 6)
    disgust_count = sum(1 for r in rows if r['emotions'][5] == 1)
    print(f"    Neutral samples: {neutral_count:,} | Disgust samples: {disgust_count:,}")
except Exception as e:
    print(f"    GoEmotions failed ({e}), skipping.")
    df_go = pd.DataFrame(columns=['text', 'emotions', 'sentiment'])

# ── MERGE & SPLIT ──────────────────────────────────────────────
print("\nMerging all datasets...")
final_df = pd.concat([df_w, df_i, df_go], ignore_index=True)
final_df = final_df.dropna()
final_df = final_df[final_df['text'].str.len() >= 10]
final_df['sentiment'] = final_df['sentiment'].astype(int)
# Deduplicate across datasets
final_df = final_df.drop_duplicates(subset=['text']).reset_index(drop=True)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Combined (after dedup): {len(final_df):,} samples")

print("\nEmotion distribution:")
all_vecs  = np.stack(final_df['emotions'].values)
for i, name in enumerate(EMOTION_LABELS):
    count = int(all_vecs[:, i].sum())
    print(f"  {name:10s}: {count:,}")

if len(final_df) == 0:
    raise ValueError("No samples!")

train_df, temp_df = train_test_split(final_df, test_size=0.30, random_state=42,
                                      stratify=final_df['sentiment'])
val_df,  test_df  = train_test_split(temp_df,  test_size=0.50, random_state=42,
                                      stratify=temp_df['sentiment'])

print("\n" + "="*60)
print("DATASET READY!")
print("="*60)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print("\nSentiment Distribution (Train):")
for idx, count in train_df['sentiment'].value_counts().sort_index().items():
    print(f"  {['Negative','Neutral','Positive'][int(idx)]}: {count:,}")

## 5. Create DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class EmotionDataset(Dataset):
    """Pre-tokenizes ALL texts once at construction — much faster than per-item."""
    def __init__(self, texts, sentiments, emotions, tokenizer, max_len=256):
        self.sentiments = [int(s) for s in sentiments]
        self.emotions   = [np.array(e, dtype=np.float32) for e in emotions]
        print(f"  Pre-tokenizing {len(texts):,} samples...", end=' ')
        encodings = tokenizer(
            [str(t) for t in texts],
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        self.input_ids      = encodings['input_ids']
        self.attention_masks = encodings['attention_mask']
        print("done.")

    def __len__(self):
        return len(self.sentiments)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'sentiment':      torch.tensor(self.sentiments[idx], dtype=torch.long),
            'emotions':       torch.tensor(self.emotions[idx],   dtype=torch.float32)
        }

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

MAX_LEN    = 256
BATCH_SIZE = 32

print("Building datasets (pre-tokenizing):")
train_dataset = EmotionDataset(train_df['text'].values, train_df['sentiment'].values,
                                list(train_df['emotions'].values), tokenizer, MAX_LEN)
val_dataset   = EmotionDataset(val_df['text'].values,   val_df['sentiment'].values,
                                list(val_df['emotions'].values),   tokenizer, MAX_LEN)
test_dataset  = EmotionDataset(test_df['text'].values,  test_df['sentiment'].values,
                                list(test_df['emotions'].values),  tokenizer, MAX_LEN)

# num_workers=0 is fine now since no tokenization happens in workers
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"\nDataLoaders ready!")
print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


## 6. Initialize Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = EnhancedSentimentModel().to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal params: {total:,} ({total/1e6:.1f}M)")
print(f"Trainable: {trainable:,} ({trainable/1e6:.1f}M)")

In [ ]:
criterion_sentiment = nn.CrossEntropyLoss()

# pos_weight for BCEWithLogitsLoss
train_vecs = np.stack(train_df['emotions'].values)
pos_counts = train_vecs.sum(axis=0).clip(min=1)
neg_counts = len(train_vecs) - pos_counts
pos_weight = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(device)
print("Emotion pos_weights:")
for name, w in zip(EMOTION_LABELS, pos_weight.cpu().tolist()):
    print(f"  {name:10s}: {w:.2f}")
criterion_emotion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Layer-wise learning rates ──────────────────────────────────────────────
# BERT encoder:  2e-5  (fine-tuning a pretrained model — stay small)
# Task heads:    1e-4  (trained from scratch — can move faster)
bert_params  = list(model.bert.parameters())
head_params  = [p for n, p in model.named_parameters() if 'bert' not in n]
optimizer = torch.optim.AdamW([
    {'params': bert_params, 'lr': 2e-5},
    {'params': head_params, 'lr': 1e-4},
], weight_decay=0.01)

scaler      = torch.amp.GradScaler('cuda')
num_epochs  = 10
PATIENCE    = 3   # early stopping patience
total_steps = len(train_loader) * num_epochs

# OneCycleLR — set separate max_lr per param group
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[6e-5, 3e-4],     # BERT group, Head group
    total_steps=total_steps
)

print(f"\nEpochs: {num_epochs} | Patience: {PATIENCE} | Total steps: {total_steps}")
print("Mixed Precision: Enabled")
print("Layer-wise LR — BERT: 2e-5 / Heads: 1e-4")


## 7. Training

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss    = float('inf')
no_improve_count = 0

print("\n" + "="*70)
print("TRAINING: BERT + CNN + BiLSTM + Attention")
print("="*70)
print(f"Samples: {len(train_df):,} | Batch: {BATCH_SIZE} | Seq: {MAX_LEN}")
print("="*70 + "\n")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-"*40)

    # ── TRAIN ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0
    train_preds, train_labels = [], []

    pbar = tqdm(train_loader, desc='Training')
    for batch in pbar:
        input_ids   = batch['input_ids'].to(device)
        mask        = batch['attention_mask'].to(device)
        sent_labels = batch['sentiment'].to(device)
        emo_labels  = batch['emotions'].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            sent_logits, emo_logits, _ = model(input_ids, mask)
            loss = (0.6 * criterion_sentiment(sent_logits, sent_labels) +
                    0.4 * criterion_emotion(emo_logits, emo_labels))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item()
        preds = torch.argmax(sent_logits, 1).cpu()
        train_preds.extend(preds.numpy())
        train_labels.extend(sent_labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = train_loss / len(train_loader)
    train_acc      = accuracy_score(train_labels, train_preds)

    # ── VALIDATE ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validating'):
            input_ids   = batch['input_ids'].to(device)
            mask        = batch['attention_mask'].to(device)
            sent_labels = batch['sentiment'].to(device)
            emo_labels  = batch['emotions'].to(device)
            with torch.amp.autocast('cuda'):
                sent_logits, emo_logits, _ = model(input_ids, mask)
                loss = (0.6 * criterion_sentiment(sent_logits, sent_labels) +
                        0.4 * criterion_emotion(emo_logits, emo_labels))
            val_loss += loss.item()
            preds = torch.argmax(sent_logits, 1).cpu()
            val_preds.extend(preds.numpy())
            val_labels.extend(sent_labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc      = accuracy_score(val_labels, val_preds)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.1%}")
    print(f"Val   Loss: {avg_val_loss:.4f} | Acc: {val_acc:.1%}")

    # ── Early stopping ─────────────────────────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss    = avg_val_loss
        no_improve_count = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print("  ✓ Best model saved!")
    else:
        no_improve_count += 1
        print(f"  ✗ No improvement ({no_improve_count}/{PATIENCE})")
        if no_improve_count >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}!")
            break

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"Best Val Loss: {best_val_loss:.4f}")
print(f"Final Val Acc: {history['val_acc'][-1]:.1%}")
print(f"Epochs run   : {len(history['val_acc'])} / {num_epochs}")


## 8b. Per-Emotion Threshold Calibration
Find the best decision threshold for each emotion on the **validation** set (avoids data leakage).

In [ ]:
# Load best model weights for calibration
model.load_state_dict(torch.load('best_model.pth', map_location=device, weights_only=True))
model.eval()

import numpy as np
from sklearn.metrics import f1_score

# Collect val probabilities
all_emo_probs  = []
all_emo_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids   = batch['input_ids'].to(device)
        mask        = batch['attention_mask'].to(device)
        emo_labels  = batch['emotions']
        with torch.amp.autocast('cuda'):
            _, emo_logits, _ = model(input_ids, mask)
        probs = torch.sigmoid(emo_logits.float()).cpu().numpy()
        all_emo_probs.append(probs)
        all_emo_labels.append(emo_labels.numpy())

all_emo_probs  = np.vstack(all_emo_probs)   # (N, 7)
all_emo_labels = np.vstack(all_emo_labels)  # (N, 7)

# Search best threshold per emotion (maximise F1)
thresholds = np.arange(0.20, 0.70, 0.05)
BEST_THRESHOLDS = []

print("Per-emotion threshold calibration:")
print(f"{'Emotion':12s} {'Best τ':>8s} {'F1 @ τ':>10s} {'F1 @ 0.5':>10s}")
print("-" * 44)
for i, name in enumerate(EMOTION_LABELS):
    best_f1, best_t = 0, 0.5
    for t in thresholds:
        preds = (all_emo_probs[:, i] >= t).astype(int)
        f1 = f1_score(all_emo_labels[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    # Baseline F1 at 0.5
    base_preds = (all_emo_probs[:, i] >= 0.5).astype(int)
    base_f1    = f1_score(all_emo_labels[:, i], base_preds, zero_division=0)
    BEST_THRESHOLDS.append(float(best_t))
    print(f"{name:12s} {best_t:>8.2f} {best_f1:>10.4f} {base_f1:>10.4f}")

print(f"\nBEST_THRESHOLDS = {BEST_THRESHOLDS}")

## 8. Test Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pth', map_location=device, weights_only=True))
model.eval()

test_preds, test_labels = [], []
test_emo_probs, test_emo_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        input_ids   = batch['input_ids'].to(device)
        mask        = batch['attention_mask'].to(device)
        sent_labels = batch['sentiment'].to(device)
        emo_labels  = batch['emotions'].to(device)
        with torch.amp.autocast('cuda'):
            sent_logits, emo_logits, _ = model(input_ids, mask)
        preds = torch.argmax(sent_logits, 1).cpu()
        test_preds.extend(preds.numpy())
        test_labels.extend(sent_labels.cpu().numpy())
        probs = torch.sigmoid(emo_logits.float()).cpu().numpy()
        test_emo_probs.append(probs)
        test_emo_labels.append(emo_labels.cpu().numpy())

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                               classification_report, hamming_loss, f1_score)

# ── Sentiment results ──────────────────────────────────────────────────────
acc  = accuracy_score(test_labels, test_preds)
prec, rec, f1, _ = precision_recall_fscore_support(test_labels, test_preds,
                                                     average='weighted')
print("\n" + "="*70)
print("TEST RESULTS — SENTIMENT")
print("="*70)
print(f"Accuracy : {acc:.4f} ({acc:.1%})")
print(f"Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
print("\n" + classification_report(test_labels, test_preds,
                                   target_names=['Negative','Neutral','Positive']))

# ── Emotion results — calibrated thresholds ────────────────────────────────
test_emo_probs  = np.vstack(test_emo_probs)    # (N, 7)
test_emo_labels = np.vstack(test_emo_labels)   # (N, 7)

# Apply per-emotion calibrated thresholds (from cell 8b)
test_emo_preds = np.zeros_like(test_emo_probs, dtype=int)
for i, t in enumerate(BEST_THRESHOLDS):
    test_emo_preds[:, i] = (test_emo_probs[:, i] >= t).astype(int)

hl = hamming_loss(test_emo_labels, test_emo_preds)
macro_f1 = f1_score(test_emo_labels, test_emo_preds, average='macro', zero_division=0)

print("\n" + "="*70)
print("TEST RESULTS — EMOTIONS  (calibrated thresholds)")
print("="*70)
print(f"Hamming Loss : {hl:.4f}  (lower is better, 0 = perfect)")
print(f"Macro F1     : {macro_f1:.4f}")
print(f"\n{'Emotion':12s} {'F1':>8s} {'Precision':>10s} {'Recall':>8s} {'Threshold':>10s}")
print("-" * 52)
for i, name in enumerate(EMOTION_LABELS):
    p = test_emo_probs[:, i]
    l = test_emo_labels[:, i]
    pr = test_emo_preds[:, i]
    f  = f1_score(l, pr, zero_division=0)
    prec_e = (pr & l.astype(int)).sum() / pr.sum() if pr.sum() > 0 else 0
    rec_e  = (pr & l.astype(int)).sum() / l.sum()  if l.sum()  > 0 else 0
    print(f"{name:12s} {f:>8.4f} {prec_e:>10.4f} {rec_e:>8.4f} {BEST_THRESHOLDS[i]:>10.2f}")

## 9b. Visualisations — Confusion Matrix & Emotion F1


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Sentiment Confusion Matrix ─────────────────────────────────────────────
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Negative','Neutral','Positive'],
            yticklabels=['Negative','Neutral','Positive'])
axes[0].set_title('Sentiment — Confusion Matrix', fontsize=13)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

# ── Per-Emotion F1 Bar Chart ───────────────────────────────────────────────
emo_f1s = [f1_score(test_emo_labels[:, i], test_emo_preds[:, i], zero_division=0)
           for i in range(NUM_EMOTIONS)]
colors = ['#4CAF50' if f >= 0.6 else '#FF9800' if f >= 0.4 else '#F44336'
          for f in emo_f1s]
bars = axes[1].barh(EMOTION_LABELS, emo_f1s, color=colors)
axes[1].set_xlim(0, 1)
axes[1].axvline(x=macro_f1, color='navy', linestyle='--',
                label=f'Macro F1 = {macro_f1:.3f}')
axes[1].set_title('Per-Emotion F1 Score', fontsize=13)
axes[1].set_xlabel('F1 Score')
axes[1].legend()
for bar, val in zip(bars, emo_f1s):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_results.png', dpi=150)
plt.show()
print("Saved model_results.png")


## 9c. Attention Visualisation
Shows which tokens the BiLSTM-Attention layer focused on for a prediction.


In [ ]:
def visualize_attention(text):
    """Visualise per-token attention weights alongside prediction."""
    model.eval()
    tokens = tokenizer.tokenize(text)[:MAX_LEN - 2]
    tokens = ['[CLS]'] + tokens + ['[SEP]']

    enc = tokenizer(text, max_length=MAX_LEN, padding='max_length',
                    truncation=True, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    mask      = enc['attention_mask'].to(device)

    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            sent_logits, emo_logits, attn_weights = model(input_ids, mask)

    # attn_weights shape: (1, seq_len) — extract actual token region
    attn = attn_weights[0].cpu().float().numpy()
    n    = len(tokens)
    attn = attn[:n]
    attn = attn / (attn.sum() + 1e-9)  # renormalise

    # Prediction
    sent_probs = torch.softmax(sent_logits.float(), 1)[0].cpu().numpy()
    emo_probs  = torch.sigmoid(emo_logits.float())[0].cpu().numpy()
    pred_sent  = ['Negative','Neutral','Positive'][int(sent_probs.argmax())]
    detected   = [(EMOTION_LABELS[i], float(emo_probs[i]))
                  for i, t in enumerate(BEST_THRESHOLDS) if emo_probs[i] >= t]
    if not detected:
        detected = [(EMOTION_LABELS[int(emo_probs.argmax())], float(emo_probs.max()))]
    detected.sort(key=lambda x: x[1], reverse=True)

    # Plot
    fig, ax = plt.subplots(figsize=(max(10, n * 0.45), 2.5))
    bars = ax.bar(range(n), attn, color=plt.cm.Reds(attn / attn.max()))
    ax.set_xticks(range(n))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    title = (f'Attention Weights  |  Sentiment: {pred_sent} ({sent_probs.max():.0%})  '
             f'|  Emotions: {" ".join([e for e,_ in detected[:2]])}')
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Attention weight')
    plt.tight_layout()
    plt.savefig('attention_viz.png', dpi=120)
    plt.show()

# Try on a few literature examples
visualize_attention("A deep melancholy settled over him as he gazed upon the ruins of what once was his home.")
visualize_attention("The report was filed. Nothing more, nothing less. The case was closed.")


## 9. Sample Predictions

In [ ]:
def predict(text, thresholds=None):
    """Predict sentiment + emotions. Uses calibrated per-emotion thresholds."""
    if thresholds is None:
        thresholds = BEST_THRESHOLDS  # calibrated on val set

    model.eval()
    enc = tokenizer(text, max_length=MAX_LEN, padding='max_length',
                    truncation=True, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    mask      = enc['attention_mask'].to(device)

    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            sent_logits, emo_logits, attn = model(input_ids, mask)

    # Sentiment
    sent_probs  = torch.softmax(sent_logits.float(), 1)[0].cpu().numpy()
    sent_names  = ['Negative', 'Neutral', 'Positive']
    pred_sent   = sent_names[int(np.argmax(sent_probs))]
    confidence  = float(sent_probs.max())

    # Emotions — apply per-emotion calibrated thresholds
    emo_probs   = torch.sigmoid(emo_logits.float())[0].cpu().numpy()
    detected    = [(EMOTION_LABELS[i], float(emo_probs[i]))
                   for i, t in enumerate(thresholds) if emo_probs[i] >= t]

    # Sort by probability; always show at least the top prediction
    if not detected:
        detected = [(EMOTION_LABELS[int(np.argmax(emo_probs))], float(emo_probs.max()))]
    detected.sort(key=lambda x: x[1], reverse=True)

    print(f"\nText     : {text[:90]}{'...' if len(text)>90 else ''}")
    print(f"Sentiment: {pred_sent} ({confidence:.1%})")
    print(f"Emotions : {', '.join([f'{e}({p:.2f})' for e,p in detected])}")
    print("-" * 60)
    return {'sentiment': pred_sent, 'confidence': confidence, 'emotions': detected}


examples = [
    "The protagonist felt overwhelming joy as she reunited with her family.",
    "A deep melancholy settled over him as he gazed upon the ruins.",
    "Fury coursed through her veins when she discovered the betrayal.",
    "The dark corridor filled her with inexplicable dread.",
    "To her astonishment, the letter contained unexpected news.",
    "The story unfolded in a matter-of-fact manner.",           # Neutral
    "He turned away in utter revulsion at the sight.",         # Disgust
    "She felt a creeping shame wash over her as they stared.", # Disgust/Shame
    "The report was filed. Nothing more, nothing less.",       # Neutral
]

print("\n" + "="*60)
print("SAMPLE PREDICTIONS")
print("="*60)
for text in examples:
    predict(text)

## 10. Training History

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], 'o-', label='Train')
ax1.plot(history['val_loss'], 's-', label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], 'o-', label='Train')
ax2.plot(history['val_acc'], 's-', label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

print("Saved training_history.png")

## 11. Save Model

In [ ]:
model_cpu = model.cpu()
torch.save({
    'model_state_dict':  model_cpu.state_dict(),
    'tokenizer':         'bert-base-uncased',
    'max_len':           MAX_LEN,
    'test_accuracy':     acc,
    'test_f1':           f1,
    'hamming_loss':      hl,
    'best_thresholds':   BEST_THRESHOLDS,           # calibrated per-emotion thresholds
    'emotion_labels':    EMOTION_LABELS,             # uses variable, not hardcoded
    'sentiment_labels':  ['Negative', 'Neutral', 'Positive'],
    'dataset':           'Wagih_ISEAR_GoEmotions'
}, 'enhanced_emotion_model.pth')

print("\n" + "="*60)
print("MODEL SAVED!")
print("="*60)
print(f"Test Accuracy   : {acc:.1%}")
print(f"Test F1         : {f1:.4f}")
print(f"Hamming Loss    : {hl:.4f}")
print(f"Emotion Macro F1: {macro_f1:.4f}")
print(f"Thresholds saved: {BEST_THRESHOLDS}")
print(f"Total samples   : {len(final_df):,}")
print("\nDownload enhanced_emotion_model.pth")
print("="*60)
